# Notebook 01 — EDA e Qualidade de Dados

**Objetivo:** Carregar os dados, explorar distribuições, validar coerência por rail (PIX/Card/Wire) e criar feature store inicial.

**Saída esperada (Dia 1):**
- ✅ Dicionário de dados (mapeamento ao schema canônico)
- ✅ Checagens de qualidade (valores faltantes, outliers, tipos)
- ✅ Coerência por rail (validação de campos específicos por canal)
- ✅ Feature store inicial (agregações por cliente, merchant, device, IP)
- ✅ Planilha no Google Sheets com achados-chave

## Setup e Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

from src.config import SEED, DATA_RAW, DATA_PROCESSED, OUTPUTS

warnings.filterwarnings('ignore')
np.random.seed(SEED)
sns.set_style('whitegrid')

print(f"✓ Seed fixada: {SEED}")
print(f"✓ Caminhos: RAW={DATA_RAW}, PROCESSED={DATA_PROCESSED}")

## 1. Carregar Dados

In [ ]:
# Carregar arquivo Excel
excel_file = DATA_RAW / "AML Case Cloudwalk INC.xlsx"

# Verificar quais sheets existem
xls = pd.ExcelFile(excel_file)
print("Sheets encontrados:")
for i, sheet in enumerate(xls.sheet_names, 1):
    print(f"  {i}. {sheet}")

In [ ]:
# Carregar cada sheet (ajuste os nomes conforme os sheets reais)
# TODO: Substituir 'transacoes', 'clientes', 'merchants' pelos nomes reais dos sheets

transactions = pd.read_excel(excel_file, sheet_name='transacoes')  # Ajustar nome
clients = pd.read_excel(excel_file, sheet_name='clientes')          # Ajustar nome
merchants = pd.read_excel(excel_file, sheet_name='merchants')       # Ajustar nome

print(f"✓ Transações: {transactions.shape}")
print(f"✓ Clientes: {clients.shape}")
print(f"✓ Merchants: {merchants.shape}")

## 2. Dicionário de Dados

Mapear colunas reais para o schema canônico (seção 4.3 do balizador)

In [ ]:
# Inspecionar colunas de transações
print("Colunas em 'transactions':")
print(transactions.columns.tolist())
print(f"\nPrimeiras linhas:")
transactions.head()

In [ ]:
# Inspecionar colunas de clientes
print("Colunas em 'clients':")
print(clients.columns.tolist())
print(f"\nPrimeiras linhas:")
clients.head()

In [ ]:
# Inspecionar colunas de merchants
print("Colunas em 'merchants':")
print(merchants.columns.tolist())
print(f"\nPrimeiras linhas:")
merchants.head()

In [ ]:
# TODO: Mapear colunas reais para schema canônico
# Exemplo:
# schema_mapping = {
#     'transactions': {'col_real_1': 'transaction_id', 'col_real_2': 'client_id', ...},
#     'clients': {'col_real_1': 'client_id', 'col_real_2': 'monthly_income', ...},
#     'merchants': {...}
# }

# Criar tabela de mapeamento
mapping_df = pd.DataFrame({
    'schema_canonical': list(transactions.columns),
    'coluna_real': list(transactions.columns),
    'tipo_esperado': [''] * len(transactions.columns),
    'presente': ['✓'] * len(transactions.columns)
})

print("Mapeamento de colunas (COMPLETAR):")
mapping_df

## 3. Checagens de Qualidade

In [ ]:
# Função auxiliar para relatório de qualidade
def quality_check(df, name):
    print(f"\n{'='*60}")
    print(f"Qualidade: {name}")
    print(f"{'='*60}")
    
    print(f"\n📊 Dimensões: {df.shape[0]} linhas × {df.shape[1]} colunas")
    
    print(f"\n❌ Valores faltantes:")
    missing = df.isnull().sum()
    missing_pct = (missing / len(df) * 100).round(2)
    missing_df = pd.DataFrame({'coluna': missing.index, 'faltantes': missing.values, '%': missing_pct.values})
    missing_df = missing_df[missing_df['faltantes'] > 0].sort_values('faltantes', ascending=False)
    if len(missing_df) == 0:
        print("  ✓ Nenhum valor faltante")
    else:
        print(missing_df.to_string(index=False))
    
    print(f"\n📈 Tipos de dados:")
    print(df.dtypes)
    
    print(f"\n🔢 Estatísticas descritivas (núméricos):")
    print(df.describe())

quality_check(transactions, "Transactions")
quality_check(clients, "Clients")
quality_check(merchants, "Merchants")

## 4. Coerência por Rail (PIX/Card/Wire)

Validar que campos específicos só existem em rails corretos

In [ ]:
# TODO: Ajustar nome da coluna 'rail' conforme dados reais
# Verificar valores únicos
if 'rail' in transactions.columns:
    print("Rails encontrados:")
    print(transactions['rail'].value_counts())
else:
    print("⚠️ Coluna 'rail' não encontrada. Procurar por: payment_method, channel, tipo_pagamento, etc")
    print(f"Colunas disponíveis: {transactions.columns.tolist()}")

In [ ]:
# Validações por rail (COMPLETAR conforme dados reais)
# Exemplo: Card deve ter 'installments' e 'eci', PIX deve ter 'counterparty_id'

def validate_rail_coherence(df):
    issues = []
    
    # TODO: Adicionar validações específicas
    # if 'rail' in df.columns and 'installments' in df.columns:
    #     card_no_installments = (df[df['rail'] == 'card']['installments'].isnull()).sum()
    #     if card_no_installments > 0:
    #         issues.append(f"  ⚠️ {card_no_installments} transações Card sem 'installments'")
    
    if len(issues) == 0:
        print("✓ Todas as validações de rail passaram")
    else:
        print("❌ Problemas encontrados:")
        for issue in issues:
            print(issue)

validate_rail_coherence(transactions)

## 5. Exploração Básica (EDA)

In [ ]:
# Período de dados
if 'timestamp' in transactions.columns or 'data' in transactions.columns:
    date_col = 'timestamp' if 'timestamp' in transactions.columns else 'data'
    transactions[date_col] = pd.to_datetime(transactions[date_col])
    print(f"Período: {transactions[date_col].min()} até {transactions[date_col].max()}")
    print(f"Duração: {(transactions[date_col].max() - transactions[date_col].min()).days} dias")
else:
    print("⚠️ Coluna de data não encontrada")

In [ ]:
# Distribuição de valores (amount)
if 'amount' in transactions.columns or 'valor' in transactions.columns:
    amount_col = 'amount' if 'amount' in transactions.columns else 'valor'
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    axes[0].hist(transactions[amount_col], bins=50, edgecolor='black')
    axes[0].set_xlabel('Valor')
    axes[0].set_ylabel('Frequência')
    axes[0].set_title('Distribuição de Valores')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].boxplot(transactions[amount_col])
    axes[1].set_ylabel('Valor')
    axes[1].set_title('Boxplot - Detecção de Outliers')
    
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'figuras' / '01_distribuicao_valores.png', dpi=100, bbox_inches='tight')
    plt.show()
    
    print(f"Estatísticas de Valor:")
    print(transactions[amount_col].describe())

In [ ]:
# Número de clientes, merchants, transações únicos
print("Entidades únicas:")
if 'client_id' in transactions.columns:
    print(f"  • Clientes: {transactions['client_id'].nunique()}")
if 'merchant_id' in transactions.columns:
    print(f"  • Merchants: {transactions['merchant_id'].nunique()}")
if 'device_id' in transactions.columns:
    print(f"  • Devices: {transactions['device_id'].nunique()}")
if 'ip_address' in transactions.columns or 'ip' in transactions.columns:
    ip_col = 'ip_address' if 'ip_address' in transactions.columns else 'ip'
    print(f"  • IPs: {transactions[ip_col].nunique()}")
print(f"  • Transações: {len(transactions)}")

## 6. Feature Store Inicial

Agregações por cliente, merchant, device, IP

In [ ]:
# TODO: Implementar feature store conforme os dados
# Exemplo:

# Agregações por cliente
if 'client_id' in transactions.columns and 'amount' in transactions.columns:
    client_aggs = transactions.groupby('client_id').agg({
        'amount': ['sum', 'mean', 'count', 'std', 'min', 'max'],
    }).round(2)
    
    client_aggs.columns = ['_'.join(col).strip() for col in client_aggs.columns.values]
    client_aggs = client_aggs.reset_index()
    
    print("Feature Store - Por Cliente (primeiras 10 linhas):")
    print(client_aggs.head(10))
    
    # Salvar
    client_aggs.to_csv(DATA_PROCESSED / 'client_features.csv', index=False)
    print(f"\n✓ Salvo em: {DATA_PROCESSED / 'client_features.csv'}")

In [ ]:
# Agregações por merchant (se houver)
if 'merchant_id' in transactions.columns and 'amount' in transactions.columns:
    merchant_aggs = transactions.groupby('merchant_id').agg({
        'amount': ['sum', 'mean', 'count'],
    }).round(2)
    
    merchant_aggs.columns = ['_'.join(col).strip() for col in merchant_aggs.columns.values]
    merchant_aggs = merchant_aggs.reset_index()
    
    print("Feature Store - Por Merchant (primeiras 10 linhas):")
    print(merchant_aggs.head(10))
    
    # Salvar
    merchant_aggs.to_csv(DATA_PROCESSED / 'merchant_features.csv', index=False)
    print(f"\n✓ Salvo em: {DATA_PROCESSED / 'merchant_features.csv'}")

## 7. Relatório de Qualidade Final

In [ ]:
# Gerar relatório resumido
quality_report = {
    'métrica': [
        'Período',
        'Clientes únicos',
        'Merchants únicos',
        'Transações',
        'Valor total',
        'Valor médio',
        'Valor máximo',
        'Valores faltantes',
        'Coerência Rails'
    ],
    'resultado': [
        'TODO',
        'TODO',
        'TODO',
        'TODO',
        'TODO',
        'TODO',
        'TODO',
        'TODO',
        '✓ Validado'
    ]
}

report_df = pd.DataFrame(quality_report)
print("\n" + "="*60)
print("RELATÓRIO DE QUALIDADE - DIA 1")
print("="*60)
print(report_df.to_string(index=False))

# Salvar para Google Sheets
report_df.to_csv(OUTPUTS / '01_quality_report.csv', index=False)
print(f"\n✓ Salvo em: {OUTPUTS / '01_quality_report.csv'}")

## 8. Próximos Passos (Dia 2)

- ✅ Implementar as ≥15 regras do catálogo
- ✅ Rodar motor de regras
- ✅ Gerar alertas + exemplos por ID
- ✅ Criar ranking de risco por regra